# Projeto 3 - Alegoria da Caverna

### Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

### Inicializando janela

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 1400
largura = 1400

window = glfw.create_window(largura, altura, "Alegoria da Caverna", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [3]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


### Carregando Modelos (vértices e texturas) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [4]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []

#Iluminação
global normals_list
normals_list = []



def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        #carregando vértices
        if values[0] == 'v':
            vertices.append([
                float(values[1]),
                float(values[2]),
                float(values[3])
            ])

        #carregando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append([
                float(values[1]),
                float(values[2])
            ])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    img = Image.open(img_textura).convert("RGB")
    
    img_width = img.size[0]
    img_height = img.size[1]
    
    image_data = img.tobytes("raw", "RGB", 0, -1)
    
    # importante para imagens RGB
    glPixelStorei(GL_UNPACK_ALIGNMENT, 1)
    
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
    
        vertices_triangulados = circular_sliding_window_of_three(face[0])
        texturas_trianguladas = circular_sliding_window_of_three(face[1])
    
        for i in range(0, len(vertices_triangulados), 3):
            v1_id = vertices_triangulados[i]
            v2_id = vertices_triangulados[i + 1]
            v3_id = vertices_triangulados[i + 2]
    
            p1 = glm.vec3(*modelo['vertices'][v1_id - 1])
            p2 = glm.vec3(*modelo['vertices'][v2_id - 1])
            p3 = glm.vec3(*modelo['vertices'][v3_id - 1])
    
            # normal da face
            normal = glm.normalize(glm.cross(p2 - p1, p3 - p1))
    
            # adiciona os 3 vértices
            vertices_list.append(modelo['vertices'][v1_id - 1])
            vertices_list.append(modelo['vertices'][v2_id - 1])
            vertices_list.append(modelo['vertices'][v3_id - 1])
    
            # adiciona as 3 coordenadas de textura
            for j in range(3):
                texture_id = texturas_trianguladas[i + j]
    
                if texture_id > 0 and texture_id <= len(modelo['texture']):
                    textures_coord_list.append(modelo['texture'][texture_id - 1])
                else:
                    textures_coord_list.append([0.0, 0.0])
    
            # mesma normal para os 3 vértices do triângulo
            for j in range(3):
                normals_list.append([normal.x, normal.y, normal.z])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    textureId = numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1

    return verticeInicial, verticeFinal - verticeInicial, textureId

def adiciona_normais(normal, quantidade=6):
    global normals_list

    for i in range(quantidade):
        normals_list.append(normal)


### Criando o cubo: skybox cubemap

Criando para cada plano (xy, xz e zy) uma função que cria um retângulo texturizado.

In [5]:
def add_textured_plane_xz(x0, x1, z0, z1, y, repeat_u=1.0, repeat_v=1.0):
    '''
    Cria um plano fixado no y (piso e teto)
    '''
    
    global vertices_list
    global textures_coord_list

    vertice_inicial = len(vertices_list)

    #definição dos vértices do retângulo
    p1 = [x0, y, z0]
    p2 = [x1, y, z0]
    p3 = [x1, y, z1]
    p4 = [x0, y, z1]

    # aplicação da imagem ao plano, parâmetros repeat indicam quantas vezes a textura repete em cada direção (mapeamento planar)
    t1 = [0.0, 0.0]
    t2 = [repeat_u, 0.0]
    t3 = [repeat_u, repeat_v]
    t4 = [0.0, repeat_v]

    vertices_list.append(p1)
    vertices_list.append(p2)
    vertices_list.append(p3)

    textures_coord_list.append(t1)
    textures_coord_list.append(t2)
    textures_coord_list.append(t3)

    vertices_list.append(p1)
    vertices_list.append(p3)
    vertices_list.append(p4)

    textures_coord_list.append(t1)
    textures_coord_list.append(t3)
    textures_coord_list.append(t4)

    vertice_final = len(vertices_list)
    quantidade_vertices = vertice_final - vertice_inicial

    adiciona_normais([0.0, 1.0, 0.0])

    return vertice_inicial, quantidade_vertices


def add_floor_sky_textured(x0, x1, z0, z1, y, arq_textura, repeat_u=1.0, repeat_v=1.0):
    '''Função cria o plano xz e carrega as texturas, retornando os parâmetros necessários para desenhar o plano'''
    inicio, qtd = add_textured_plane_xz(
        x0, x1,
        z0, z1,
        y,
        repeat_u,
        repeat_v
    )

    global numberTextures
    floor_sky_texture = numberTextures

    load_texture_from_file(floor_sky_texture, arq_textura)
    numberTextures += 1

    return inicio, qtd, floor_sky_texture

In [6]:
def add_textured_plane_xy(x0, x1, y0, y1, z, repeat_u=1.0, repeat_v=1.0):
    '''
    Cria um plano fixado no z (frente e trás)
    '''
    
    global vertices_list
    global textures_coord_list

    vertice_inicial = len(vertices_list)

    #definição dos vértices do retângulo
    p1 = [x0, y0, z]
    p2 = [x1, y0, z]
    p3 = [x1, y1, z]
    p4 = [x0, y1, z]

    # aplicação da imagem ao plano, parâmetros repeat indicam quantas vezes a textura repete em cada direção (mapeamento planar)
    t1 = [0.0, 0.0]
    t2 = [repeat_u, 0.0]
    t3 = [repeat_u, repeat_v]
    t4 = [0.0, repeat_v]

    vertices_list.append(p1)
    vertices_list.append(p2)
    vertices_list.append(p3)

    textures_coord_list.append(t1)
    textures_coord_list.append(t2)
    textures_coord_list.append(t3)

    vertices_list.append(p1)
    vertices_list.append(p3)
    vertices_list.append(p4)

    textures_coord_list.append(t1)
    textures_coord_list.append(t3)
    textures_coord_list.append(t4)

    vertice_final = len(vertices_list)

    adiciona_normais([0.0, 0.0, 1.0])
    return vertice_inicial, vertice_final - vertice_inicial

def add_top_bottom_textured(x0, x1, y0, y1, z, arq_textura, repeat_u=1.0, repeat_v=1.0):
    '''Função cria o plano xy e carrega as texturas, retornando os parâmetros necessários para desenhar o plano'''

    inicio, qtd = add_textured_plane_xy(
        x0, x1,
        y0, y1,
        z,
        repeat_u,
        repeat_v
    )

    global numberTextures
    top_bottom_texture = numberTextures

    load_texture_from_file(top_bottom_texture, arq_textura)
    numberTextures += 1

    return inicio, qtd, top_bottom_texture

In [7]:
def add_textured_plane_yz(y0, y1, z0, z1, x, repeat_u=1.0, repeat_v=1.0):
    '''
    Cria um plano fixado no x (esquerda e direita)
    '''
    
    global vertices_list
    global textures_coord_list

    vertice_inicial = len(vertices_list)

    #definição dos vértices do retângulo
    p1 = [x, y0, z0]
    p2 = [x, y0, z1]
    p3 = [x, y1, z1]
    p4 = [x, y1, z0]

    # aplicação da imagem ao plano, parâmetros repeat indicam quantas vezes a textura repete em cada direção (mapeamento planar)
    t1 = [0.0, 0.0]
    t2 = [repeat_u, 0.0]
    t3 = [repeat_u, repeat_v]
    t4 = [0.0, repeat_v]

    vertices_list.append(p1)
    vertices_list.append(p2)
    vertices_list.append(p3)

    textures_coord_list.append(t1)
    textures_coord_list.append(t2)
    textures_coord_list.append(t3)

    vertices_list.append(p1)
    vertices_list.append(p3)
    vertices_list.append(p4)

    textures_coord_list.append(t1)
    textures_coord_list.append(t3)
    textures_coord_list.append(t4)

    vertice_final = len(vertices_list)

    adiciona_normais([1.0, 0.0, 0.0])
    return vertice_inicial, vertice_final - vertice_inicial

def add_right_left_textured(y0, y1, z0, z1, x, arq_textura, repeat_u=1.0, repeat_v=1.0):
    '''Função cria o plano yz e carrega as texturas, retornando os parâmetros necessários para desenhar o plano'''

    inicio, qtd = add_textured_plane_yz(
        y0, y1,
        z0, z1,
        x,
        repeat_u,
        repeat_v
    )

    global numberTextures
    right_left_texture = numberTextures

    load_texture_from_file(right_left_texture, arq_textura)
    numberTextures += 1

    return inicio, qtd, right_left_texture

In [8]:
def carregar_textura_ceu(pasta, arquivo):
    global numberTextures

    texture_id = numberTextures
    load_texture_from_file(texture_id, f"cenario/{pasta}/{arquivo}")
    numberTextures += 1

    return texture_id

### Vamos carregar cada modelo e definir funções para desenhá-los

#### Criando o cenário externo

Definição do skybox e dos dois planos que simulam a praia externa (areia e mar)

In [9]:
verticeInicial_areia, qtdVertices_areia, textureId_areia = add_floor_sky_textured(
    -15, 15, -15, 15, -1.0, "cenario/areia.jpg", 8.0, 8.0
)

def desenha_areia():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_areia)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_areia, qtdVertices_areia) ## renderizando

verticeInicial_mar, qtdVertices_mar, textureId_mar = add_floor_sky_textured(
     -20, 20, -20, 20, -1.03, "cenario/mar.png", 1.0, -1.0
)

def desenha_mar():
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_mar)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_mar, qtdVertices_mar) ## renderizando

#Definição do cubo

xmin = -30
xmax =  30
zmin = -30
zmax =  30
ymin = -2
ymax =  28

verticeInicial_bottom, qtdVertices_bottom, textureId_bottom = add_floor_sky_textured(
     -30, 30, -30, 30, -1.05, "cenario/ceu_noite/bottom.png", 1.0, 1.0
)

def desenha_bottom(textureId_bottom):
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_bottom)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_bottom, qtdVertices_bottom) ## renderizando

verticeInicial_ceu, qtdVertices_ceu, textureId_ceu = add_floor_sky_textured(
    xmin, xmax, zmin, zmax, ymax, "cenario/ceu_noite/top.png", 1.0, -1.0
)

def desenha_ceu_top(textureId_ceu):
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_ceu)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_ceu, qtdVertices_ceu) ## renderizando

verticeInicial_right, qtdVertices_right, textureId_right = add_right_left_textured(
    ymin, ymax, zmin, zmax, xmax, "cenario/ceu_noite/right.png", -1.0, 1.0
)

def desenha_ceu_right(textureId_right):
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_right)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_right, qtdVertices_right) ## renderizando

verticeInicial_left, qtdVertices_left, textureId_left = add_right_left_textured(
    ymin, ymax, zmin, zmax, xmin, "cenario/ceu_noite/left.png", 1.0, 1.0
)

def desenha_ceu_left(textureId_left):
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_left)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_left, qtdVertices_left) ## renderizando


verticeInicial_front, qtdVertices_front, textureId_front = add_top_bottom_textured(
    xmin, xmax, ymin, ymax, zmax, "cenario/ceu_noite/front.png", 1.0, 1.0
)

def desenha_ceu_front(textureId_front):
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_front)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_front, qtdVertices_front) ## renderizando

verticeInicial_back, qtdVertices_back, textureId_back = add_top_bottom_textured(
    xmin, xmax, ymin, ymax, zmin, "cenario/ceu_noite/back.png", -1.0, 1.0
)

def desenha_ceu_back(textureId_back):
    mat_model = np.array(glm.mat4(1.0)) #Matriz identidade
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_back)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_back, qtdVertices_back) ## renderizando

0
1
2
3
4
5
6
7


In [10]:
# Carregando todas as texturas de céu que serão utilizadasaa
ceu_noite = {
    "bottom": carregar_textura_ceu("ceu_noite", "bottom.png"),
    "top": carregar_textura_ceu("ceu_noite", "top.png"),
    "front": carregar_textura_ceu("ceu_noite", "front.png"),
    "back": carregar_textura_ceu("ceu_noite", "back.png"),
    "right": carregar_textura_ceu("ceu_noite", "right.png"),
    "left": carregar_textura_ceu("ceu_noite", "left.png")
}

ceu_nascer = {
    "bottom": carregar_textura_ceu("ceu_nascer", "bottom.png"),
    "top": carregar_textura_ceu("ceu_nascer", "top.png"),
    "front": carregar_textura_ceu("ceu_nascer", "front.png"),
    "back": carregar_textura_ceu("ceu_nascer", "back.png"),
    "right": carregar_textura_ceu("ceu_nascer", "right.png"),
    "left": carregar_textura_ceu("ceu_nascer", "left.png")
}

ceu_azul = {
    "bottom": carregar_textura_ceu("ceu_azul", "bottom.png"),
    "top": carregar_textura_ceu("ceu_azul", "top.png"),
    "front": carregar_textura_ceu("ceu_azul", "front.png"),
    "back": carregar_textura_ceu("ceu_azul", "back.png"),
    "right": carregar_textura_ceu("ceu_azul", "right.png"),
    "left": carregar_textura_ceu("ceu_azul", "left.png")
}

ceu_por = {
    "bottom": carregar_textura_ceu("ceu", "bottom.png"),
    "top": carregar_textura_ceu("ceu", "top.png"),
    "front": carregar_textura_ceu("ceu", "front.png"),
    "back": carregar_textura_ceu("ceu", "back.png"),
    "right": carregar_textura_ceu("ceu", "right.png"),
    "left": carregar_textura_ceu("ceu", "left.png")
}

8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31


#### Carregando objetos do cenário

In [11]:
verticeInicial_caverna, quantosVertices_caverna, textureId_caverna = load_obj_and_texture('objetos/caverna/GRUTA_BASE.obj', ['objetos/caverna/GRUTA_BASE_defaultMat_BaseColor.png'])
verticeInicial_muro, qtdVertices_muro, textureId_muro = load_obj_and_texture('objetos/muro/muro.obj', ['objetos/muro/muro.png'])
verticeInicial_montePedras,qtdVertices_montePedras, textureId_montePedras = load_obj_and_texture('objetos/pedra_circulo/pedra_circulo.obj', ['objetos/pedra_circulo/pedra_circulo.png']) 
verticeInicial_palmeira, quantosVertices_palmeira, textureId_palmeira = load_obj_and_texture('objetos/palmeira/palm.obj', ['objetos/palmeira/palmTexture.jpg'])
verticeInicial_rocha, quantosVertices_rocha, textureId_rocha = load_obj_and_texture('objetos/rocha/Didier_rock_2.obj', ['objetos/rocha/Didier_Rock_2_Albedo.png'])
verticeInicial_concha, quantosVertices_concha, textureId_concha = load_obj_and_texture('objetos/concha/shell.obj', ['objetos/concha/shell.jpg'])
verticeInicial_fogueira, qtdVertices_fogueira, textureId_fogueira = load_obj_and_texture('objetos/fogueira/fogueira.obj', ['objetos/fogueira/Koster_Base_Color.png'])
verticeInicial_fire, qtdVertices_fire, textureId_fire = load_obj_and_texture('objetos/fire/fire.obj', ['objetos/fire/fire.png'])

verticeInicial_prisioneiro, qtdVertices_prisioneiro, textureId_prisioneiro = load_obj_and_texture('objetos/prisioneiro/prisioneiro.obj', ['objetos/prisioneiro/prisioneiro.png'])
verticeInicial_guarda, qtdVertices_guarda, textureId_guarda = load_obj_and_texture('objetos/guarda/guarda.obj', ['objetos/guarda/guarda_base.png'])

verticeInicial_cavalo, qtdVertices_cavalo, textureId_cavalo = load_obj_and_texture('objetos/estatua_cavalo/estatua_cavalo.obj', ['objetos/estatua_cavalo/base_color.jpg'])
verticeInicial_ibex, qtdVertices_ibex, textureId_ibex = load_obj_and_texture('objetos/ibex_estatua/estatua_ibex.obj', ['objetos/ibex_estatua/base_ibex.jpeg'])

verticeInicial_sol, qtdVertices_sol, textureId_sol = load_obj_and_texture('objetos/sol/sol.obj', ['objetos/sol/sol_textura.jpg'])


global verticeInicial_fogueira, qtdVertices_fogueira

def desenha_caverna(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_caverna)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_caverna, quantosVertices_caverna) ## renderizando

def desenha_muro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_muro)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_muro, qtdVertices_muro) ## renderizando
    
def desenha_montePedras(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId_montePedras)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_montePedras, qtdVertices_montePedras) ## renderizando

def desenha_fogueira(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_fogueira)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_fogueira, qtdVertices_fogueira) ## renderizando

def desenha_fire(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    cx, cy, cz = centro_objeto(verticeInicial_fire, qtdVertices_fire)
    
    mat_model = model_com_centro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, cx, cy, cz)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)
    
    glBindTexture(GL_TEXTURE_2D, textureId_fire)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_fire, qtdVertices_fire) ## renderizando

def desenha_prisioneiro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)

    glBindTexture(GL_TEXTURE_2D, textureId_prisioneiro)

    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_prisioneiro, qtdVertices_prisioneiro) ## renderizando

def desenha_guarda(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)

    glBindTexture(GL_TEXTURE_2D, textureId_guarda)

    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_guarda, qtdVertices_guarda) ## renderizando

def desenha_cavalo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)

    glBindTexture(GL_TEXTURE_2D, textureId_cavalo)

    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_cavalo, qtdVertices_cavalo) ## renderizando

def desenha_ibex(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_usa_textura = glGetUniformLocation(program, "usa_textura")
    glUniform1i(loc_usa_textura, True)

    glBindTexture(GL_TEXTURE_2D, textureId_ibex)

    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_ibex, qtdVertices_ibex) ## renderizando

def desenha_palmeira(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
               
    # usa a textura da palmeira
    glBindTexture(GL_TEXTURE_2D, textureId_palmeira)
    
    # desenha
    glDrawArrays(GL_TRIANGLES, verticeInicial_palmeira, quantosVertices_palmeira)

def desenha_rocha(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    # usa a textura da palmeira
    glBindTexture(GL_TEXTURE_2D, textureId_rocha)
    
    # desenha
    glDrawArrays(GL_TRIANGLES, verticeInicial_rocha, quantosVertices_rocha)
    
def desenha_concha(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    # usa a textura da palmeira
    glBindTexture(GL_TEXTURE_2D, textureId_concha)
    
    # desenha
    glDrawArrays(GL_TRIANGLES, verticeInicial_concha, quantosVertices_concha)


def desenha_sol(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    # usa a textura da palmeira
    glBindTexture(GL_TEXTURE_2D, textureId_sol)
    
    # desenha
    glDrawArrays(GL_TRIANGLES, verticeInicial_sol, qtdVertices_sol)

Processando modelo objetos/caverna/GRUTA_BASE.obj. Vertice inicial: 48
Processando modelo objetos/caverna/GRUTA_BASE.obj. Vertice final: 30462
32
Processando modelo objetos/muro/muro.obj. Vertice inicial: 30462
Processando modelo objetos/muro/muro.obj. Vertice final: 199566
33
Processando modelo objetos/pedra_circulo/pedra_circulo.obj. Vertice inicial: 199566
Processando modelo objetos/pedra_circulo/pedra_circulo.obj. Vertice final: 229533
34
Processando modelo objetos/palmeira/palm.obj. Vertice inicial: 229533
Processando modelo objetos/palmeira/palm.obj. Vertice final: 424359
35
Processando modelo objetos/rocha/Didier_rock_2.obj. Vertice inicial: 424359
Processando modelo objetos/rocha/Didier_rock_2.obj. Vertice final: 445902
36
Processando modelo objetos/concha/shell.obj. Vertice inicial: 445902
Processando modelo objetos/concha/shell.obj. Vertice final: 1045902
37
Processando modelo objetos/fogueira/fogueira.obj. Vertice inicial: 1045902
Processando modelo objetos/fogueira/fogueira

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar dois slots (buffers): um para os vértices e outro para as texturas.

In [12]:
buffer_VBO = glGenBuffers(3)

### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [13]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [14]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### Enviando as normais para a GPU

In [15]:
normals = np.zeros(len(normals_list), [("position", np.float32, 3)])
normals['position'] = normals_list

glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])
glBufferData(GL_ARRAY_BUFFER, normals.nbytes, normals, GL_STATIC_DRAW)

stride = normals.strides[0]
offset = ctypes.c_void_p(0)

loc_normal = glGetAttribLocation(program, "normal")
glEnableVertexAttribArray(loc_normal)
glVertexAttribPointer(loc_normal, 3, GL_FLOAT, False, stride, offset)

### Eventos para modificar a posição da câmera.

* Usei as teclas A, S, D e W para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [16]:
#cameraPos   = glm.vec3(0.0,  0.0,  1.0);
#cameraFront = glm.vec3(0.0,  0.0, -1.0);
#cameraUp    = glm.vec3(0.0,  1.0,  0.0);

def limitar_camera():
    '''A função tem o objetivo de limitar a movimentação da câmera pela cena, o limite de todas as direções se dá pelo cubo definido no skybox e pelo piso da caverna'''
    
    global xmin, xmax, ymax, zmin, zmax #dimensões do cubo
    y_min = 1.4 #piso da caverna

    #limitação da câmera em todos os eixos
    cameraPos.x = max(xmin, min(cameraPos.x, xmax-4)) 
    cameraPos.y = max(y_min, min(cameraPos.y, ymax-1))
    cameraPos.z = max(zmin, min(cameraPos.z, zmax-4))
    
# camera
cameraPos   = glm.vec3(1.5, 0.9, -10)
cameraFront = glm.vec3(0.0, 0.0, 1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   = -90.0	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0

firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2

def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp
    global polygonal_mode, apagar_fogo, tz, vento_ligado
    global luz_sol_ligada, luz_fogo_ligada, luz_ambiente_ligada
    global intensidade_ambiente, difusa_global
    global angulo_sol

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    
    cameraSpeed = 50 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    if key==glfw.KEY_F and action == glfw.PRESS: #controla o fogo
        apagar_fogo = not apagar_fogo 

    if key == glfw.KEY_V and action == glfw.PRESS: #controla o vento
        vento_ligado = not vento_ligado

    if key == glfw.KEY_R: #reset
        tz = 0

    # translação dos guardas
    if key == glfw.KEY_RIGHT: 
        tz+=0.01

    if key == glfw.KEY_LEFT:
        tz-=0.01

    if key == glfw.KEY_UP:
        angulo_sol += 0.2
    
    if key == glfw.KEY_DOWN:
        angulo_sol -= 0.2

    #Controle do ângulo do sol
    if angulo_sol > 190:
        angulo_sol = 0        

    # 1 liga/desliga a luz do sol
    if key == glfw.KEY_1 and action == glfw.PRESS:
        luz_sol_ligada = not luz_sol_ligada
    
    # 2 liga/desliga a luz da fogueira
    if key == glfw.KEY_2 and action == glfw.PRESS:
        luz_fogo_ligada = not luz_fogo_ligada
    
    # 3 liga/desliga a luz ambiente
    if key == glfw.KEY_3 and action == glfw.PRESS:
        luz_ambiente_ligada = not luz_ambiente_ligada
    
    # Z diminui a luz ambiente
    if key == glfw.KEY_Z and action == glfw.PRESS:
        intensidade_ambiente = max(0.0, intensidade_ambiente - 0.05)
    
    # X aumenta a luz ambiente
    if key == glfw.KEY_X and action == glfw.PRESS:
        intensidade_ambiente = min(1.0, intensidade_ambiente + 0.05)
    
    # C diminui a reflexão difusa global
    if key == glfw.KEY_C and action == glfw.PRESS:
        difusa_global = max(0.0, difusa_global - 0.05)
    
    # B aumenta a reflexão difusa global
    if key == glfw.KEY_B and action == glfw.PRESS:
        difusa_global = min(2.0, difusa_global + 0.05)

    limitar_camera()

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Funções

In [17]:
def centro_objeto(inicio, qtd):
    '''Função para calcular o centro de um objeto - usar como referência para as transformações geometricas'''
    xs, ys, zs = [], [], []

    for i in range(inicio, inicio + qtd):
        x, y, z = vertices_list[i]
        xs.append(float(x))
        ys.append(float(y))
        zs.append(float(z))

    cx = sum(xs) / len(xs)
    cy = sum(ys) / len(ys)
    cz = sum(zs) / len(zs)
    return cx, cy, cz

### Matrizes Model, View e Projection

In [18]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    #gera a matriz M = T · R · S
    
    return matrix_transform

def model_com_centro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, c_x, c_y, c_z):
    angle = math.radians(angle)

    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade

    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))

    matrix_transform = glm.translate(matrix_transform, glm.vec3(c_x, c_y, c_z)) #colocando tanto rotação como escala com referência ao centro do objeto

    # aplicando rotacao (segunda operação a ser executada)
    if angle != 0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))

    # aplicando escala (primeira operação a ser executada) com referência ao centro do objeto 
    
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    matrix_transform = glm.translate(matrix_transform, glm.vec3(-c_x, -c_y, -c_z))

    matrix_transform = np.array(matrix_transform)

    #gera a matriz M = T · T(c) · R ·  S · T(-c)
    
    return matrix_transform
    

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 100.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### funções auxiliares para enviar uniforms

In [19]:
def mistura_cor(cor1, cor2, t):
    """
    Mistura linearmente duas cores.
    t = 0 retorna cor1.
    t = 1 retorna cor2.
    """
    return (
        cor1[0] * (1 - t) + cor2[0] * t,
        cor1[1] * (1 - t) + cor2[1] * t,
        cor1[2] * (1 - t) + cor2[2] * t
    )


def cor_sol_por_angulo(angulo):
    """
    Define a cor da luz do sol de acordo com o ângulo.

    0 a 30 graus: nascer do sol, luz avermelhada indo para branca.
    30 a 150 graus: dia, luz branca.
    150 a 180 graus: pôr do sol, luz branca indo para avermelhada.
    """

    cor_vermelha = (1.0, 0.35, 0.12)
    cor_branca = (1.0, 1.0, 0.92)

    if angulo <= 30:
        t = angulo / 30.0
        return mistura_cor(cor_vermelha, cor_branca, t)
    elif angulo <= 150:
        return cor_branca
    elif angulo <= 180:
        t = (angulo - 150.0) / 30.0
        return mistura_cor(cor_branca, cor_vermelha, t)
    else:
        return cor_vermelha

def textura_ceu_por_angulo(angulo):
    if angulo <= 30:
        return ceu_nascer
    elif angulo <= 150:
        return ceu_azul
    elif angulo <= 180:
        return ceu_por
    else:
        return ceu_noite

In [20]:
def set_bool(nome, valor):
    loc = glGetUniformLocation(program, nome)
    glUniform1i(loc, int(valor))

def set_float(nome, valor):
    loc = glGetUniformLocation(program, nome)
    glUniform1f(loc, valor)

def set_vec3(nome, x, y, z):
    loc = glGetUniformLocation(program, nome)
    glUniform3f(loc, x, y, z)

In [21]:
def configurar_material(
    recebe_sol,
    recebe_fogo,
    recebe_iluminacao=True,
    ka=(1.0, 1.0, 1.0),
    kd=0.8,
    ks=0.3,
    shininess=32.0
):
    set_bool("recebeSol", recebe_sol)
    set_bool("recebeFogo", recebe_fogo)
    set_bool("recebeIluminacao", recebe_iluminacao)

    set_vec3("materialAmbient", ka[0], ka[1], ka[2])
    set_float("materialDiffuse", kd)
    set_float("materialSpecular", ks)
    set_float("materialShininess", shininess)

### Nesse momento, nós exibimos a janela!


In [22]:
glfw.show_window(window)

### Loop principal da janela.

In [23]:
glEnable(GL_DEPTH_TEST) ### importante para 3D
polygonal_mode = False 

#Controle do fogo
global apagar_fogo
apagar_fogo = False 

angulo_sol = -1
tz = 0

#Controle das palmeiras
global vento_ligado
vento_ligado = True

palmeiras = [] #lista de palmeiras

QUANTIDADE = 16 #qtd de palmeiras

luz_sol_ligada = True
luz_fogo_ligada = True
luz_ambiente_ligada = True

intensidade_ambiente = 0.25
difusa_global = 1.0

def longe_de_outras(x, z, lista, min_dist=2):
    for (px, _, pz, _) in lista:
        if ((x - px)**2 + (z - pz)**2)**0.5 < min_dist:
            return False
    return True

while len(palmeiras) < QUANTIDADE:
    
    # metade esquerda ou direita
    if random.random() < 0.5:
        x = random.uniform(-12, -5)
    else:
        x = random.uniform(5, 13)

    z = random.uniform(-12, 14)
    y = -1

    if longe_de_outras(x, z, palmeiras):
        escala = random.uniform(0.008, 0.015)
        palmeiras.append((x, y, z, escala))


while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events()

    glClearColor(1.0, 1.0, 1.0, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)
    
    # calcula posição do sol
    raio = 27
    x_sol = raio * math.cos(math.radians(angulo_sol))
    y_sol = raio * math.sin(math.radians(angulo_sol))
    z_sol = 0

    r_sol, g_sol, b_sol = cor_sol_por_angulo(angulo_sol)
    
    # envia luzes
    set_vec3("viewPos", cameraPos.x, cameraPos.y, cameraPos.z)
    
    set_bool("luzSolLigada", luz_sol_ligada)
    set_bool("luzFogoLigada", luz_fogo_ligada and not apagar_fogo)
    set_bool("luzAmbienteLigada", luz_ambiente_ligada)
    
    set_float("intensidadeAmbiente", intensidade_ambiente)
    set_float("difusaGlobal", difusa_global)
    
    set_vec3("solPos", x_sol, y_sol, z_sol)
    set_vec3("solColor", r_sol, g_sol, b_sol)
    set_vec3("fogoPos", 7.1, 0.5, -4.4)
    
    #Cena
    # areia
    configurar_material(
        recebe_sol=True,
        recebe_fogo=False,
        recebe_iluminacao=True,
        ka=(1.0, 1.0, 1.0),
        kd=0.9,
        ks=0.1,
        shininess=16.0
    )
    desenha_areia()
    
    # mar
    configurar_material(
        recebe_sol=True,
        recebe_fogo=False,
        recebe_iluminacao=True,
        ka=(0.8, 0.9, 1.0),
        kd=0.8,
        ks=0.7,
        shininess=64.0
    )
    desenha_mar()
    
    # céu sem iluminação
    configurar_material(
        recebe_sol=False,
        recebe_fogo=False,
        recebe_iluminacao=False
    )
    
    texturas_ceu_atual = textura_ceu_por_angulo(angulo_sol)

    desenha_bottom(texturas_ceu_atual["bottom"])
    desenha_ceu_top(texturas_ceu_atual["top"])
    
    desenha_ceu_right(texturas_ceu_atual["right"])
    desenha_ceu_left(texturas_ceu_atual["left"])
    desenha_ceu_front(texturas_ceu_atual["front"])
    desenha_ceu_back(texturas_ceu_atual["back"])

    configurar_material(
        recebe_sol=True,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.5, 0.5, 0.5),
        kd=0.6,
        ks=0.1,
        shininess=8.0
    )
    desenha_caverna(0.0, 0.0, 0, 1, 0.5, 1.4, -5, 2.5, 2, -2.5)
    
    # =========================
    # OBJETOS INTERNOS
    # Recebem luz da fogueira, mas NÃO recebem luz do sol
    # =========================
    
    # muro
    configurar_material(
        recebe_sol=False,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.6, 0.6, 0.6),
        kd=0.65,
        ks=0.15,
        shininess=16.0
    )
    desenha_muro(90, 0, 1, 0, 0.9, -0.92, 0, 1, 1, 1)
    
    
    # monte de pedras próximo da fogueira
    configurar_material(
        recebe_sol=False,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.5, 0.5, 0.5),
        kd=0.7,
        ks=0.1,
        shininess=8.0
    )
    
    add_y = 0
    for i in range(8):
        desenha_montePedras(
            0, 0, 1, 0,
            2.8, -0.75 + add_y, -4,
            0.5, 0.5, 0.5
        )
        add_y += 0.1


    # estrutura da fogueira
    configurar_material(
        recebe_sol=False,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.7, 0.45, 0.25),
        kd=0.8,
        ks=0.2,
        shininess=24.0
    )
    desenha_fogueira(0, 0, 0, 0, 7.1, 0, -4.4, 0.8, 0.8, 0.8)

    
    # fogo visual
    cx, cy, cz = centro_objeto(verticeInicial_fogueira, qtdVertices_fogueira)
    
    if apagar_fogo:
        sx_fire = 0
        sy_fire = 0
        sz_fire = 0
    else:
        sx_fire = 100 + 8 * math.sin(4 * glfw.get_time())
        sy_fire = 100
        sz_fire = 100 + 8 * math.sin(4 * glfw.get_time())
    
    # O fogo em si não precisa receber iluminação.
    # Ele deve aparecer como objeto emissivo/visual.
    configurar_material(
        recebe_sol=False,
        recebe_fogo=False,
        recebe_iluminacao=False
    )
    
    desenha_fire(
        0, 0, 0, 0,
        cx + 8.1, cy + 0.2, cz - 4.6,
        sx_fire, sy_fire, sz_fire
    )


    # prisioneiros
    configurar_material(
        recebe_sol=False,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.65, 0.65, 0.65),
        kd=0.75,
        ks=0.25,
        shininess=32.0
    )

    add_z = 0
    for i in range(2):
        desenha_prisioneiro(
            270, 0, 1, 0,
            -0.49, -0.86, 2.0 + add_z,
            1.1, 1.1, 1.1
        )
        add_z -= 1.5


    # guardas
    configurar_material(
        recebe_sol=False,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.6, 0.6, 0.6),
        kd=0.7,
        ks=0.35,
        shininess=48.0
    )
    
    add_z = 0
    for i in range(4):
        desenha_guarda(
            180, 0, 1, 0,
            1.6, -0.8, 2.0 + add_z + tz,
            0.01, 0.01, 0.01
        )
        add_z -= 1.5


    # estátua do cavalo
    configurar_material(
        recebe_sol=False,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.8, 0.8, 0.8),
        kd=0.6,
        ks=0.6,
        shininess=64.0
    )

    desenha_cavalo(
        90, 0, 1, 0,
        1.315, 1.25, 0.755 + tz,
        1.3, 1.3, 1.3
    )
    
    
    # estátua do ibex
    configurar_material(
        recebe_sol=False,
        recebe_fogo=True,
        recebe_iluminacao=True,
        ka=(0.8, 0.8, 0.8),
        kd=0.6,
        ks=0.6,
        shininess=64.0
    )
    
    desenha_ibex(
        -45, 0, 1, 0,
        1.315, 1.25, 0.8 - 1.5 + tz,
        0.4, 0.4, 0.4
    )
    
    
    # =========================
    # OBJETOS EXTERNOS
    # Recebem luz do sol, mas NÃO recebem luz da fogueira
    # =========================
    
    # palmeiras
    configurar_material(
        recebe_sol=True,
        recebe_fogo=False,
        recebe_iluminacao=True,
        ka=(0.7, 0.9, 0.7),
        kd=0.85,
        ks=0.15,
        shininess=16.0
    )
    
    tempo = glfw.get_time()
    
    for i, (x, y, z, escala) in enumerate(palmeiras):
    
        if vento_ligado:
            angle = 5 * math.sin(2 * tempo + i)
        else:
            angle = 0
    
        desenha_palmeira(
            angle,
            1, 0, 0,
            x, y, z,
            escala, escala, escala
        )
        
    
    # rochas externas
    configurar_material(
        recebe_sol=True,
        recebe_fogo=False,
        recebe_iluminacao=True,
        ka=(0.55, 0.55, 0.55),
        kd=0.75,
        ks=0.2,
        shininess=20.0
    )
    
    desenha_rocha(0, 0, 1, 0, -13, -1.2, 9, 4, 4, 4)
    desenha_rocha(0, 0, 1, 0, 3, -1.2, 12, 4, 4, 4)
    desenha_rocha(0, 0, 1, 0, 12, -1.2, -13, 4, 4, 4)
    desenha_rocha(0, 0, 1, 0, -13, -1.2, -13, 4, 4, 4)


    # conchas externas
    configurar_material(
        recebe_sol=True,
        recebe_fogo=False,
        recebe_iluminacao=True,
        ka=(0.9, 0.85, 0.75),
        kd=0.8,
        ks=0.5,
        shininess=64.0
    )
    
    desenha_concha(-90, 1, 0, 0, 15, -0.8, -1, 0.007, 0.007, 0.007)
    desenha_concha(-90, 1, 0, 0, -13, -1, 13, 0.007, 0.007, 0.007)
    
    desenha_concha(-90, 1, 0, 0, -1, -0.8, 11, 0.007, 0.007, 0.007)
    desenha_concha(-90, 1, 0, 0, 0, -0.8, 10, 0.007, 0.007, 0.007)
    desenha_concha(-90, 1, 0, 0, -0.5, -0.8, 12, 0.007, 0.007, 0.007)
    desenha_concha(-90, 1, 0, 0, -1, -0.8, 13, 0.007, 0.007, 0.007)
    desenha_concha(-90, 1, 0, 0, 0, -0.8, 14, 0.007, 0.007, 0.007)
    # desenha_sol (0, 0, 1, 0, 12, 25, -13, 0.002, 0.002, 0.002)

    
    configurar_material(
        recebe_sol=False,
        recebe_fogo=False,
        recebe_iluminacao=False
    )
    desenha_sol(
        0,          
        0,0,1,
        x_sol, y_sol, z_sol,
        0.002,0.002,0.002
    )
    
    glfw.swap_buffers(window)

glfw.terminate()